In [28]:
import requests 
from bs4 import BeautifulSoup as bs
import pandas as pd
import tldextract
import numpy as np
import matplotlib.pylab as plt
import os
import re

In [29]:
def on_sale_chk(text):
    if len(text)<1:
        return False
    return 'domain' in text and 'sale' in text

def on_parked_chk(text):
    if len(text)<1:
        return True
    return 'domain' in text and 'park' in text

def on_Parked(text):
    if len(text)<1:
        return True
    return (('website' in text or 'content' in text) and 'unavailable' in text) or ('will' in text and 'soon' in text)

In [30]:
#returns html contents, textual character length, website size, status code, parked or on sale

def soupFromUrl(scrapeUrl):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 6.1; zh-CN) AppleWebKit/533+ (KHTML, like Gecko)'}
    try:
        req = requests.get(scrapeUrl, headers=headers, timeout=5)
        # print(req.status_code)
        req.close()
        if req.status_code == 200:
            # print(bs(req.text, 'html.parser').get_text().strip().replace('\n',' '))
            soup = bs(req.text,'html')

            # print(soup)

            text = ''

            if soup.body:
                text = re.sub(r'[^\w]', ' ',soup.body.get_text(' ', strip=True).lower())

            # print(soup)

            # print('text',text)
            # print(bs(req.text, 'html.parser'))
            # return [bs(req.text, 'html.parser'),len(req.text), len(req.content), req.status_code]
            # print([len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))])
            return [len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))]
        else:
            # return [-1,0,0,req.status_code]
            return [0,0,req.status_code,0]
    except:
        # return [-1,0,0,-1]
        return [0,0,-1,0]

In [31]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 6.1; zh-CN) AppleWebKit/533+ (KHTML, like Gecko)'}
req = requests.get('https://www.lycos.com/', headers=headers, timeout=5)
print(req.status_code)
req.close()
if req.status_code == 200:
    # print(bs(req.text, 'html.parser').get_text().strip().replace('\n',' '))
    soup = bs(req.text,'html')

    # print(soup)
    text = ''

    if soup.body:
        text = re.sub(r'[^\w]', ' ',soup.body.get_text(' ', strip=True).lower())

    print(soup)
    print('text',text)
    print([len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))])

200
<!DOCTYPE html>

<html lang="en">
<head>
<meta charset="utf-8"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<!-- The above 3 meta tags *must* come first in the head; any other head content must come *after* these tags -->
<meta content="Lycos, Inc., is a web search engine and web portal established in 1994, spun out of Carnegie Mellon University. Lycos also encompasses a network of email, webhosting, social networking, and entertainment websites." name="description"/>
<meta content="" name="author"/>
<link href="https://ly.lygo.net/static/lycos/img/favicon.ico" rel="icon" type="image/png"/>
<title>Lycos.com</title>
<link href="//fonts.googleapis.com/css?family=Lato:400,300,300italic,400italic,700,700italic" rel="stylesheet" type="text/css"/>
<link href="/css/in/fonts.css" rel="stylesheet" type="text/css">
<link href="https://ly.lygo.net/static/lycos/css/in/font-awesome.css" rel="stylesheet" type="text

In [32]:
# print(soupFromUrl('https://www.delinian.com/'))
# print(soupFromUrl('https://www.makecashonline.com/'))
print(soupFromUrl('https://www.lycos.com/'))

[12956, 328, 200, 0]


In [33]:
ExAIS_url = list(pd.read_csv('../Dataset/URL Data/ExAIS_Dataset.csv',delimiter='\t')['URL'])
ExAIS_url[:10]

['http://t.co/Qpe80nnaJi',
 'http://fb.me/sms',
 'http://g.co/freezone',
 'http://bit.ly/N1saY4',
 'http://bit.ly/1b2WYiu',
 'https://fb.com/l/1NjTfumaF0i0P9M',
 'https://fb.com/l/1Oz23niDoqYHa06',
 'http://goo.gl/u1C6G',
 'http://goo.gl/u1C6G',
 'http://goo.gl/u1C6G']

In [34]:
import re

def find_first_slash_preceded_by_number(s):
    # Regular expression to find the first instance of a number followed by '/'
    match = re.search(r'\d+/', s)
    
    if match:
        return match.start() + len(match.group()) - 1  # Return the index of '/'
    else:
        return -1  # Return -1 if no match is found

# Example usage
string = "example77/test 88/test2 99/test3"
index = find_first_slash_preceded_by_number(string)
print(index)  # Outputs the index of the first '/' preceded by a number

9


In [35]:
unique_ExAIS_url = set(ExAIS_url)

In [36]:
for idx,i in enumerate(unique_ExAIS_url):
    if '..' in i:
        if 'www' in i:
            unique_ExAIS_url[idx] = ''
        else:
            unique_ExAIS_url[idx] = unique_ExAIS_url[idx].replace('..','.')

In [37]:
unique_ExAIS_url = [i for i in unique_ExAIS_url if i!='']

In [38]:
ExAIS_dataset = {'ham':list(pd.read_csv('../Dataset/URL Data/ExAIS_Dataset_Ham.csv',delimiter='\t')['URL']),'spam':list(pd.read_csv('../Dataset/URL Data/ExAIS_Dataset_Spam.csv',delimiter='\t')['URL'])}

In [39]:
ExAIS_url_in_ham = [0]*len(unique_ExAIS_url)
ExAIS_url_in_spam = [0]*len(unique_ExAIS_url)

for idx,i in enumerate(unique_ExAIS_url):
    for j in ExAIS_dataset['ham']:
        if not isinstance(j, str):
            continue
        if i in j:
            ExAIS_url_in_ham[idx] = 1
            break
    for j in ExAIS_dataset['spam']:
        if not isinstance(j, str):
            continue
        if i in j:
            ExAIS_url_in_spam[idx] = 1
            break

print(ExAIS_url_in_ham.count(1))
print(ExAIS_url_in_spam.count(1))

38
83


In [40]:
common_urls = []
for i in range(len(ExAIS_url_in_ham)):
    if ExAIS_url_in_ham[i]==ExAIS_url_in_spam[i]:
        common_urls.append(unique_ExAIS_url[i])

len(common_urls)

8

In [41]:
ExAIS_dataset['Unique Url'] = unique_ExAIS_url

In [42]:
import tldextract

def FQDN(Url):
    
    url_extract_res = tldextract.extract(Url)
    fqdn = ''
    if url_extract_res.subdomain:
        fqdn = url_extract_res.subdomain + '.' + url_extract_res.domain + '.' + url_extract_res.suffix
        # fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    else:
        fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    
    return fqdn

In [43]:
ExAIS_dataset['FQDN'] = [FQDN(i) for i in ExAIS_dataset['Unique Url']]

len(set(ExAIS_dataset['FQDN']))

66

In [44]:
a = soupFromUrl('https://facebook.com')

print(a)

[75842, 645, 200, 0]


The below query last ran on 11 Feb 2025

In [45]:
# website_size, text_content_length, status_code, parked = [],[],[],[]

# for i in ExAIS_dataset['FQDN']:
#     a = soupFromUrl('https://'+i)

#     website_size.append(a[0])
#     text_content_length.append(a[1])
#     status_code.append(a[2])
#     parked.append(a[3])


# # parked = [0]*len(ExAIS_dataset)

# # for idx,i in enumerate(ExAIS_dataset['FQDN']):
# #     if ExAIS_dataset['Status Code'][idx]==200:
# #         a = soupFromUrl('https://'+i)
# #         parked[idx] = a[3]
# #         # break

# # print(parked)

In [59]:
ExAIS_dataset = pd.read_csv('../Dataset/URL Data/ExAIS Websites Analysis.csv')

In [46]:
# ExAIS_dataset['Website Size in KB'] = website_size
# ExAIS_dataset['Website Textual Content Length'] = text_content_length
# ExAIS_dataset['Status Code'] = status_code

# ExAIS_dataset['Parked'] = parked

In [47]:
for i in ExAIS_dataset:
    print(len(ExAIS_dataset[i]))

71
138
115
115
115
115
115
115


In [48]:
ExAIS_dataset['ham'] = ExAIS_url_in_ham
ExAIS_dataset['spam'] = ExAIS_url_in_spam

In [49]:
ExAIS_dataset

{'ham': [0,
  1,
  1,
  0,
  0,
  0,
  0,
  1,
  1,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  0,
  0,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  1,
  1,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  1],
 'spam': [1,
  0,
  0,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,

In [50]:
# ExAIS_dataset = pd.read_csv('../Dataset/URL Data/ExAIS Websites Analysis.csv')
ExAIS_dataset = pd.DataFrame.from_dict(ExAIS_dataset)
ExAIS_dataset.head()

,ham,spam,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,0,1,http://glo.lifestyle.com,glo.lifestyle.com,0,0,-1,0
1,1,0,www.redcrossnigeria.org,www.redcrossnigeria.org,75325,4803,200,0
2,1,0,http://whatsapp.com/dl/,whatsapp.com,261534,4199,200,0
3,0,1,www.mtnonline.com/about-mtn/contact-us,www.mtnonline.com,0,0,-1,0
4,0,1,http://www.gloworld.com/globackup/,www.gloworld.com,0,0,403,0


In [51]:
ExAIS_dataset['Status Code'].value_counts()

Status Code
 200    70
-1      39
 403     6
Name: count, dtype: int64

In [52]:
numbers_to_replace = [501,403, 401]

# Value to replace with
new_value = 200

# Update the column
ExAIS_dataset.loc[ExAIS_dataset['Status Code'].isin(numbers_to_replace), 'Status Code'] = new_value

In [53]:
ExAIS_dataset['Status Code'].value_counts()

Status Code
 200    76
-1      39
Name: count, dtype: int64

In [54]:
print(len(ExAIS_dataset[(ExAIS_dataset['Status Code']==200) & (ExAIS_dataset['ham']==1)]))
print(len(ExAIS_dataset[(ExAIS_dataset['Status Code']==200) & (ExAIS_dataset['spam']==1)]))

20
59


In [55]:
ExAIS_dataset['Parked'].value_counts()

Parked
0    83
1    32
Name: count, dtype: int64

In [56]:
print(len(ExAIS_dataset[(ExAIS_dataset['Parked']==1) & (ExAIS_dataset['ham']==1)]))
print(len(ExAIS_dataset[(ExAIS_dataset['Parked']==1) & (ExAIS_dataset['ham']==0)]))

13
19


In [57]:
ExAIS_dataset.head()

,ham,spam,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,0,1,http://glo.lifestyle.com,glo.lifestyle.com,0,0,-1,0
1,1,0,www.redcrossnigeria.org,www.redcrossnigeria.org,75325,4803,200,0
2,1,0,http://whatsapp.com/dl/,whatsapp.com,261534,4199,200,0
3,0,1,www.mtnonline.com/about-mtn/contact-us,www.mtnonline.com,0,0,-1,0
4,0,1,http://www.gloworld.com/globackup/,www.gloworld.com,0,0,200,0


In [58]:
# ExAIS_dataset.to_csv('../Dataset/URL Data/ExAIS Websites Analysis.csv', index=None)

In [62]:
ExAIS_dataset.drop_duplicates(subset='FQDN', inplace=True)
print(len(ExAIS_dataset))
print(ExAIS_dataset['Status Code'].value_counts())
print(ExAIS_dataset['Parked'].value_counts())
print(len(ExAIS_dataset[(ExAIS_dataset['Status Code']==200) & (ExAIS_dataset['ham']==1)]))
print(len(ExAIS_dataset[(ExAIS_dataset['Status Code']==200) & (ExAIS_dataset['ham']!=1)]))
print(len(ExAIS_dataset[(ExAIS_dataset['Parked']==1) & (ExAIS_dataset['ham']==1)]))
print(len(ExAIS_dataset[(ExAIS_dataset['Parked']==1) & (ExAIS_dataset['ham']==0)]))

66
Status Code
-1      34
 200    32
Name: count, dtype: int64
Parked
0    56
1    10
Name: count, dtype: int64
9
23
3
7
